# Đề tài: SmartHVACControlByDoubleQLearning
## Notebook 02: Xây dựng Môi trường Mô phỏng (Environment) & Thiết kế State, Action, Reward

**Mục tiêu:**
- Trình bày nguyên lý nhiệt động học phòng (Resistance-Capacitance Thermal Model) dựa trên dữ liệu thực tế CU-BEMS.
- Định nghĩa không gian trạng thái (State Space) rời rạc hóa phục vụ bảng Q-Table.
- Định nghĩa không gian hành động (Action Space) tương ứng với 4 cấp độ công suất HVAC.
- Thiết kế và kiểm thử hàm phần thưởng đa mục tiêu (Multi-objective Reward Function) kết hợp giữa Tiện nghi nhiệt (Thermal Comfort) và Chi phí năng lượng (Energy Cost).
- Kiểm tra tương tác bước (Step & Reset) của môi trường `SmartHVACEnv`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.environment.hvac_env import SmartHVACEnv, ACTION_POWER_MAP, ACTION_NAMES
from src.environment.thermal_model import BuildingThermalModel
from src.reward.reward_function import HVACRewardFunction
from src.data.preprocess import discretize_state, state_to_index, index_to_state

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

### 1. Khởi tạo Mô hình Nhiệt động học (Building Thermal Dynamics Model)
Mô hình RC bậc 1 mô tả sự biến thiên nhiệt độ phòng:
$$T_{in}(t+1) = T_{in}(t) + \alpha_{amb} (T_{amb}(t) - T_{in}(t)) + \beta_{int} P_{int}(t) - \gamma_{cool} P_{hvac}(t) + \epsilon$$

In [ ]:
thermal_model = BuildingThermalModel()
print(f'Hệ số trao đổi nhiệt với môi trường ngoài alpha_amb: {thermal_model.alpha_amb}')
print(f'Hệ số truyền nhiệt nội tại từ máy móc/đèn beta_int: {thermal_model.beta_int} °C/kW')
print(f'Hệ số làm mát từ điều hòa gamma_cool: {thermal_model.gamma_cool} °C/kW')

### 2. Kiểm thử Không gian Trạng thái (State Space) & Rời rạc hóa (Discretization)

In [ ]:
# Thử nghiệm rời rạc hóa với nhiệt độ 23.5°C, phụ tải 5 kW, lúc 14 giờ chiều
state_tuple = discretize_state(temp_c=23.5, internal_heat_kw=5.0, hour=14)
state_idx = state_to_index(state_tuple)
recovered_tuple = index_to_state(state_idx)

print(f'Continuous Inputs: Temp=23.5°C, Heat=5.0kW, Hour=14')
print(f'State Tuple (TempBin, LoadBin, TimeBin): {state_tuple}')
print(f'Flat State Index (0-44): {state_idx}')
print(f'Recovered Tuple: {recovered_tuple}')

### 3. Kiểm thử Hàm Phần thưởng (Reward Function)

In [ ]:
reward_fn = HVACRewardFunction()

# Thử nghiệm 3 trường hợp: (1) Rất lạnh, (2) Thoải mái, (3) Rất nóng
test_temps = [19.0, 23.2, 28.0]
for t in test_temps:
    r, breakdown = reward_fn.calculate_reward(temp_c=t, hvac_kw=10.0, is_workhour=1, hour=10)
    print(f'Temp={t:4.1f}°C -> Reward={r:6.2f} (Penalty={breakdown["comfort_penalty"]:5.2f}, Energy={breakdown["energy_cost"]:4.2f})')

### 4. Tương tác với Môi trường `SmartHVACEnv` trong 24 giờ

In [ ]:
df_data = pd.read_csv(project_root / 'data' / 'processed' / 'cubems_zone1_hourly.csv')
env = SmartHVACEnv(df_data, episode_length=24)

state, info = env.reset(start_idx=100, initial_temp=26.5)
history = []

for t in range(24):
    # Thử nghiệm áp dụng hành động MEDIUM (22 kW) khi > 24°C, ngược lại LOW/OFF
    action = 2 if info['temp_c'] > 24.0 else 0
    next_state, reward, done, trunc, info = env.step(action)
    history.append({
        'step': info['step'],
        'temp_c': info['temp_c'],
        'hvac_kw': info['hvac_kw'],
        'reward': reward
    })

df_sim = pd.DataFrame(history)
df_sim.head(10)

### 5. Kết luận Giai đoạn 2:
- Môi trường `SmartHVACEnv` đã phản ánh chính xác các quy luật vật lý và tải nhiệt thực tế của tòa nhà CU-BEMS.
- Không gian hành động 4 mức và không gian trạng thái 45 trạng thái được chuẩn hóa, hoàn toàn sẵn sàng cho việc huấn luyện các thuật toán Học tăng cường Q-Learning và Double Q-Learning.